# PLINK / GWAS QC Workflow Agent
## Colab 01 — State + QC Plan Validator

本 Notebook 是期末專案第一階段。

### 目標
1. 定義 Agent State
2. 定義 QC Plan 格式
3. 建立 rule-based Plan Validator
4. 測試正常案例
5. 測試錯誤案例
6. 展示 `FAIL → 修正 → PASS`

這一階段先不接 LLM、不接 RAG、不需要 API Key。

## 0. 開發策略

- **Google Colab**：執行與測試
- **GitHub**：版本控制與最終成果保存
- **Google Drive**：之後若需要保存大型 index 或暫存檔再使用

先把 Validator 跑通，再接 RAG 與 Planning Agent。

In [ ]:
from copy import deepcopy
from pprint import pprint

MAX_PLAN_ITERATIONS = 3

print("Environment ready.")

## 1. Agent State

In [ ]:
def create_initial_state():
    return {
        "study_info": {},
        "retrieved_docs": [],
        "qc_plan": [],
        "plan_errors": [],
        "commands": [],
        "command_errors": [],
        "plan_iteration": 0,
        "command_iteration": 0,
        "final_report": ""
    }

state = create_initial_state()
pprint(state)

## 2. 範例 Study Information

In [ ]:
study_info = {
    "study_design": "case-control",
    "data_format": "plink_binary",
    "sample_size": 3000,
    "phenotype": "Disease Status",
    "covariates": ["Age", "Sex"],
    "need_pca": True,
    "need_relatedness": True,
    "association": "logistic"
}

state["study_info"] = study_info
pprint(state["study_info"])

## 3. 定義 QC Plan 格式

每一步包含：
- `step`
- `name`
- `purpose`
- `plink_flag`
- `example_setting`
- `note`
- `source`

In [ ]:
good_plan = [
    {
        "step": 1,
        "name": "Variant Missingness",
        "purpose": "Remove variants with excessive missing genotype calls.",
        "plink_flag": "--geno",
        "example_setting": "0.02",
        "note": "Example threshold only; adjust according to study design.",
        "source": "S02"
    },
    {
        "step": 2,
        "name": "Sample Missingness",
        "purpose": "Remove samples with excessive missing genotype calls.",
        "plink_flag": "--mind",
        "example_setting": "0.02",
        "note": "Example threshold only; adjust according to study design.",
        "source": "S02"
    },
    {
        "step": 3,
        "name": "MAF Filtering",
        "purpose": "Filter variants according to minor allele frequency.",
        "plink_flag": "--maf",
        "example_setting": "0.01",
        "note": "Example threshold only; adjust according to study design.",
        "source": "S02"
    },
    {
        "step": 4,
        "name": "HWE Filtering",
        "purpose": "Filter variants according to Hardy-Weinberg equilibrium test.",
        "plink_flag": "--hwe",
        "example_setting": "1e-6",
        "note": "Example threshold only; adjust according to study design.",
        "source": "S02"
    },
    {
        "step": 5,
        "name": "LD Pruning",
        "purpose": "Create an approximately LD-independent SNP subset.",
        "plink_flag": "--indep-pairwise",
        "example_setting": "50 5 0.2",
        "note": "Use the pruned subset for relatedness/PCA, not automatically for the final GWAS scan.",
        "source": "S03"
    },
    {
        "step": 6,
        "name": "Relatedness",
        "purpose": "Estimate pairwise relatedness / IBD.",
        "plink_flag": "--genome",
        "example_setting": None,
        "note": "Usually run on LD-pruned variants.",
        "source": "S04"
    },
    {
        "step": 7,
        "name": "PCA",
        "purpose": "Estimate population structure.",
        "plink_flag": "--pca",
        "example_setting": "10",
        "note": "Number of PCs is study-specific.",
        "source": "S05"
    },
    {
        "step": 8,
        "name": "Association Analysis",
        "purpose": "Test SNP associations with a case-control phenotype.",
        "plink_flag": "--logistic",
        "example_setting": None,
        "note": "Use the post-QC association dataset; do not automatically restrict the GWAS to LD-pruned SNPs.",
        "source": "S06"
    }
]

state["qc_plan"] = good_plan
pprint(state["qc_plan"])

## 4. QC Plan Validator

第一版檢查：
1. Coverage
2. Order / Logic
3. Source
4. Threshold wording
5. Final-GWAS pruning misuse

In [ ]:
def normalize_name(name):
    return str(name).strip().lower()

def find_step_index(plan, keyword):
    keyword = keyword.lower()
    for i, step in enumerate(plan):
        if keyword in normalize_name(step.get("name", "")):
            return i
    return None

def validate_qc_plan(study_info, plan):
    errors = []

    if not isinstance(plan, list) or len(plan) == 0:
        return ["QC plan is empty."]

    names = [normalize_name(step.get("name", "")) for step in plan]

    # A. Coverage
    if not any("variant missingness" in n for n in names):
        errors.append("Missing required step: Variant Missingness.")
    if not any("sample missingness" in n for n in names):
        errors.append("Missing required step: Sample Missingness.")
    if not any("maf" in n for n in names):
        errors.append("Missing required step: MAF Filtering.")
    if not any("hwe" in n for n in names):
        errors.append("Missing required step: HWE Filtering.")
    if not any("association" in n for n in names):
        errors.append("Missing required step: Association Analysis.")

    if study_info.get("need_relatedness", False):
        if not any("relatedness" in n for n in names):
            errors.append("Relatedness was requested but the plan has no Relatedness step.")

    if study_info.get("need_pca", False):
        if not any("pca" in n for n in names):
            errors.append("PCA was requested but the plan has no PCA step.")

    # B. Order / Logic
    ld_idx = find_step_index(plan, "ld pruning")
    rel_idx = find_step_index(plan, "relatedness")
    pca_idx = find_step_index(plan, "pca")
    assoc_idx = find_step_index(plan, "association")

    if study_info.get("need_relatedness", False) and rel_idx is not None:
        if ld_idx is None:
            errors.append("Relatedness is requested, but LD Pruning is missing.")
        elif ld_idx > rel_idx:
            errors.append("LD Pruning should occur before Relatedness.")

    if study_info.get("need_pca", False) and pca_idx is not None:
        if ld_idx is None:
            errors.append("PCA is requested, but LD Pruning is missing.")
        elif ld_idx > pca_idx:
            errors.append("LD Pruning should occur before PCA.")

        if assoc_idx is not None and pca_idx > assoc_idx:
            errors.append("PCA should occur before Association Analysis.")

    # C. Source
    for i, step in enumerate(plan, start=1):
        source = step.get("source")
        if source is None or str(source).strip() == "":
            errors.append(f"Step {i} ({step.get('name', 'Unknown')}) has no source.")

    # D. Threshold wording
    bad_phrases = [
        "standard threshold",
        "universal threshold",
        "must use",
        "fixed threshold"
    ]

    for i, step in enumerate(plan, start=1):
        combined = " ".join([
            str(step.get("example_setting", "")),
            str(step.get("note", "")),
            str(step.get("purpose", ""))
        ]).lower()

        for phrase in bad_phrases:
            if phrase in combined:
                errors.append(
                    f"Step {i} ({step.get('name', 'Unknown')}) may describe an example threshold as universal: '{phrase}'."
                )

    # E. Final-GWAS pruning misuse
    for i, step in enumerate(plan, start=1):
        if "association" in normalize_name(step.get("name", "")):
            combined = " ".join([
                str(step.get("purpose", "")),
                str(step.get("note", ""))
            ]).lower()

            risky_patterns = [
                "only ld-pruned",
                "only use ld-pruned",
                "final gwas uses pruned",
                "final gwas only"
            ]

            if any(p in combined for p in risky_patterns):
                errors.append(
                    f"Step {i} (Association Analysis) may incorrectly restrict the final GWAS to the LD-pruned SNP subset."
                )

    return errors

## 5. Test Case 1 — 正常案例

In [ ]:
state["qc_plan"] = deepcopy(good_plan)
state["plan_errors"] = validate_qc_plan(
    state["study_info"],
    state["qc_plan"]
)

if len(state["plan_errors"]) == 0:
    print("PLAN VALIDATION: PASS")
else:
    print("PLAN VALIDATION: FAIL")
    pprint(state["plan_errors"])

## 6. Test Case 2 — 故意製造錯誤

故意建立：
1. Missing Relatedness
2. PCA 在 Association 後
3. MAF threshold 被說成固定標準
4. HWE 沒有 Source
5. Final GWAS 只用 LD-pruned SNP

In [ ]:
bad_plan = deepcopy(good_plan)

# 1. Remove Relatedness
bad_plan = [x for x in bad_plan if x["name"] != "Relatedness"]

# 2. Move PCA after Association
pca_step = next(x for x in bad_plan if x["name"] == "PCA")
bad_plan = [x for x in bad_plan if x["name"] != "PCA"]
bad_plan.append(pca_step)

for i, step in enumerate(bad_plan, start=1):
    step["step"] = i

# 3. Bad threshold wording
for step in bad_plan:
    if step["name"] == "MAF Filtering":
        step["note"] = "0.01 is the standard threshold and must use this value."

# 4. Missing source
for step in bad_plan:
    if step["name"] == "HWE Filtering":
        step["source"] = None

# 5. Final GWAS misuse
for step in bad_plan:
    if step["name"] == "Association Analysis":
        step["note"] = "Final GWAS only use LD-pruned SNPs."

errors = validate_qc_plan(study_info, bad_plan)

print("PLAN VALIDATION:", "PASS" if len(errors) == 0 else "FAIL")
print()

for i, err in enumerate(errors, start=1):
    print(f"{i}. {err}")

## 7. 模擬 FAIL → 修正 → PASS

現在先用手動修正版模擬 Planner revise。
下一階段才會改成 LLM 自動修正。

In [ ]:
state["qc_plan"] = deepcopy(bad_plan)
state["plan_iteration"] = 0

while state["plan_iteration"] < MAX_PLAN_ITERATIONS:
    state["plan_errors"] = validate_qc_plan(
        state["study_info"],
        state["qc_plan"]
    )

    print(f"Iteration {state['plan_iteration'] + 1}")

    if not state["plan_errors"]:
        print("PLAN VALIDATION: PASS")
        break

    print("PLAN VALIDATION: FAIL")
    for err in state["plan_errors"]:
        print("-", err)

    state["plan_iteration"] += 1

    # 模擬 Planner 修正
    state["qc_plan"] = deepcopy(good_plan)

    print("\nPlan revised. Rechecking...\n")

state["plan_errors"] = validate_qc_plan(
    state["study_info"],
    state["qc_plan"]
)

print("\nFinal Result:")
print("PASS" if len(state["plan_errors"]) == 0 else "STOP / NEED HUMAN REVIEW")
print("Revision count:", state["plan_iteration"])

## 8. 第一階段完成標準

如果可以看到：

```text
Test Case 1 → PASS

Test Case 2 → FAIL
並列出多個 validator error

Loop → FAIL → revised → PASS
```

代表第一階段完成。

---

## 下一階段

下一個 Notebook：

**`02_rag_retrieval.ipynb`**

將加入：

1. 讀取 `wiki/*.md`
2. Chunking
3. Embedding
4. FAISS
5. 根據 Study Information retrieve relevant Wiki knowledge
6. 保留 `source` / `source_id`

之後再串：

```text
RAG
↓
Planner
↓
Plan Validator
```